# 13 · Shop Type Mix

Computes a Shannon entropy score for the mix of shop types (labels) around each establishment within a local neighbourhood radius.

Higher entropy = more diverse commercial mix nearby; lower entropy = dominated by fewer categories.

**Input:** `csv/00_base_data.csv`
**Output:** `csv/13_mix_shop_types.csv` — `osm_id`, `shop_mix_entropy`

---

### How it works

1. **Load** `csv/00_base_data.csv` (generated by notebook 01) containing all commercial establishments with `osm_id`, `lat`, `lon`, and `label` (food_drink, retail, health, etc.).
2. **Build a BallTree** with the coordinates of all establishments for efficient spatial neighbour lookup.
3. **For each establishment**, find all other establishments within **200 m**.
4. **Count the label distribution** among those neighbours and compute **Shannon entropy** (base 2):
   - 50% food_drink, 25% retail, 25% health → high entropy (~1.5 bits) → **diverse mix**
   - 95% food_drink, 5% retail → low entropy (~0.3 bits) → **dominated by one type**
   - No neighbours within 200 m → entropy = 0
5. **Save** `csv/13_mix_shop_types.csv` with `osm_id` and `shop_mix_entropy`. The orchestrator joins this by `osm_id` into the combined CSV.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
from collections import Counter

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")
print(f"\nLabel distribution:")
print(df["label"].value_counts())

In [ ]:
# ── Compute Shannon entropy of shop labels within 200 m of each shop ──

NEIGHBOUR_RADIUS_M = 200

coords_rad = np.radians(df[["lat", "lon"]].values)
tree = BallTree(coords_rad, metric="haversine")

radius_rad = NEIGHBOUR_RADIUS_M / 6_371_000  # metres → radians

neighbours = tree.query_radius(coords_rad, r=radius_rad)

labels = df["label"].values

def shannon_entropy(indices):
    """Shannon entropy (base-2) of label distribution among neighbours."""
    neighbour_labels = labels[indices]
    counts = Counter(neighbour_labels)
    total = len(neighbour_labels)
    if total <= 1:
        return 0.0
    probs = np.array(list(counts.values())) / total
    return -np.sum(probs * np.log2(probs))

df["shop_mix_entropy"] = [shannon_entropy(idx) for idx in neighbours]

print(f"Neighbourhood radius: {NEIGHBOUR_RADIUS_M} m")
print(f"\nshop_mix_entropy stats:")
print(f"  Mean : {df['shop_mix_entropy'].mean():.3f}")
print(f"  Std  : {df['shop_mix_entropy'].std():.3f}")
print(f"  Min  : {df['shop_mix_entropy'].min():.3f}")
print(f"  Max  : {df['shop_mix_entropy'].max():.3f}")

In [ ]:
df_out = df[["osm_id", "shop_mix_entropy"]]
df_out.to_csv("csv/13_mix_shop_types.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/13_mix_shop_types.csv")
print(df_out.describe().round(3))